# PixelQuery S3 UseCase Guide

S3에 저장된 위성영상을 **GDAL 없이** 시공간 검색, Polygon Clip, 통계, 시계열 분석, COG Export까지.

**아키텍처**: `COG (S3)` → `VirtualTIFF (메타만)` → `Icechunk (가상 참조)` → `xarray (lazy)` → `numpy`

**핵심**: `PixelQueryS3` 한 클래스로 모든 보일러플레이트를 감춤.

---

## 0. 환경 설정

In [3]:
import os, time, tempfile
import numpy as np
from shapely.geometry import shape
from pixelquery.io.s3_client import PixelQueryS3

print("Ready!")

Ready!


In [4]:
# 성능 측정 유틸
class Timer:
    """간단한 성능 측정기"""
    def __init__(self, label=""):
        self.label = label
        self.elapsed = 0
    def __enter__(self):
        self.t0 = time.perf_counter()
        return self
    def __exit__(self, *_):
        self.elapsed = time.perf_counter() - self.t0
        if self.label:
            print(f"  [{self.label}] {self.elapsed:.3f}s")

perf_log = {}  # 벤치마크 결과 저장

---
## 1. 클라이언트 생성 + COG 인제스트

`PixelQueryS3` 하나로 Icechunk 저장소 생성, VCC 연결, 레지스트리 설정이 모두 끝납니다.

In [5]:
with Timer("클라이언트 생성") as t:
    pq = PixelQueryS3(
        bucket="pixelquery-test",
        endpoint_url="http://localhost:9000",
        access_key_id="minioadmin",
        secret_access_key="minioadmin",
    )
perf_log["client_init"] = t.elapsed

print(f"클라이언트 생성 완료")

  2026-02-24T16:36:45.593642Z  WARN icechunk::storage::object_store: The LocalFileSystem storage is not safe for concurrent commits. If more than one thread/process will attempt to commit at the same time, prefer using object stores.
    at icechunk/src/storage/object_store.rs:81

  [클라이언트 생성] 0.142s
클라이언트 생성 완료


In [6]:
# S3 COG 목록 조회
cogs = pq.list_cogs("arps/")
sr_cogs = [c for c in cogs if "_sr." in c]

print(f"전체 COG: {len(cogs)}개, SR COG: {len(sr_cogs)}개")
for p in sr_cogs[:3]:
    print(f"  {p.split('/')[-1]}")
if len(sr_cogs) > 3:
    print(f"  ... +{len(sr_cogs)-3}개")

전체 COG: 10개, SR COG: 7개
  2025-04-06_analysis_ready_ps_sr.tiff
  2025-01-01_analysis_ready_ps_sr.tiff
  2025-11-14_analysis_ready_ps_sr.tiff
  ... +4개


In [7]:
# 인제스트 (한 줄!)
KNOWN_BOUNDS = [128.6500, 36.2995, 128.7598, 36.3326]  # 경북 지역

with Timer("Ingest") as t:
    groups = pq.ingest_cogs(
        cog_urls=sr_cogs,
        band_names=["band1", "band2", "band3", "band4"],
        bounds=KNOWN_BOUNDS,
    )
perf_log["ingest_total"] = t.elapsed
perf_log["ingest_per_cog"] = t.elapsed / len(sr_cogs)

print(f"\n인제스트 완료: {len(groups)}개 COG")
print(f"  총 시간: {t.elapsed:.2f}s")
print(f"  COG당:   {t.elapsed/len(sr_cogs)*1000:.1f}ms")

/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.


  [Ingest] 0.737s

인제스트 완료: 7개 COG
  총 시간: 0.74s
  COG당:   105.3ms


/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(


---
## 2. 시공간 메타데이터 검색

픽셀을 읽지 않고 **메타데이터만으로** 빠르게 필터링합니다.

In [8]:
# 전체 씬 조회
with Timer("list_scenes (all)") as t:
    for _ in range(100):
        all_scenes = pq.list_scenes()
perf_log["list_all"] = t.elapsed / 100

print(f"전체 씬: {len(all_scenes)}개")
for s in all_scenes[:5]:
    print(f"  {s['group']} | {s['acquisition_time'][:10]}")
if len(all_scenes) > 5:
    print(f"  ... +{len(all_scenes)-5}개")

  [list_scenes (all)] 0.073s
전체 씬: 7개
  scene_20250406_0000 | 2025-04-06
  scene_20250101_0001 | 2025-01-01
  scene_20251114_0002 | 2025-11-14
  scene_20250903_0003 | 2025-09-03
  scene_20250725_0004 | 2025-07-25
  ... +2개


In [9]:
# 시간 범위 필터
with Timer("list_scenes (time)") as t:
    for _ in range(100):
        time_filtered = pq.list_scenes(time_range=("2025-04-01", "2025-10-01"))
perf_log["list_time"] = t.elapsed / 100

print(f"2025-04 ~ 2025-09: {len(time_filtered)}개")
for s in time_filtered:
    print(f"  {s['acquisition_time'][:10]}")

  [list_scenes (time)] 0.070s
2025-04 ~ 2025-09: 5개
  2025-04-06
  2025-09-03
  2025-07-25
  2025-07-27
  2025-05-14


In [10]:
# 공간 + 시공간 AND 조건
query_bbox = (128.70, 36.31, 128.74, 36.33)

with Timer("list_scenes (spatial)") as t:
    for _ in range(100):
        spatial_filtered = pq.list_scenes(bounds=query_bbox)
perf_log["list_spatial"] = t.elapsed / 100

with Timer("list_scenes (time+spatial)") as t:
    for _ in range(100):
        combined = pq.list_scenes(time_range=("2025-04-01", "2025-10-01"), bounds=query_bbox)
perf_log["list_combined"] = t.elapsed / 100

print(f"공간 필터:   {len(spatial_filtered)}개")
print(f"시공간 AND:  {len(combined)}개")
for s in combined:
    print(f"  {s['acquisition_time'][:10]}")

  [list_scenes (spatial)] 0.069s
  [list_scenes (time+spatial)] 0.060s
공간 필터:   7개
시공간 AND:  5개
  2025-04-06
  2025-09-03
  2025-07-25
  2025-07-27
  2025-05-14


---
## 3. 씬 열기 + 픽셀 읽기

In [11]:
# Lazy open (S3 접근 없음)
with Timer("open_scene (lazy)") as t:
    for _ in range(10):
        ds = pq.open_scene(all_scenes[0])
perf_log["open_lazy"] = t.elapsed / 10

print(f"dims:   {dict(ds.sizes)}")
print(f"coords: {list(ds.coords)}")
print(f"dtype:  {ds['data'].dtype}")

# 실제 픽셀 읽기 (S3에서 fetch)
with Timer("pixel read") as t:
    data = ds["data"].values
perf_log["full_read"] = t.elapsed

mb = data.nbytes / 1024 / 1024
print(f"\nshape:      {data.shape}")
print(f"size:       {data.size:,} pixels ({mb:.1f} MB)")
print(f"throughput: {mb/t.elapsed:.0f} MB/s")
print(f"range:      [{np.nanmin(data)}, {np.nanmax(data)}]")

/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.


  [open_scene (lazy)] 5.634s
dims:   {'band': 4, 'y': 874, 'x': 3519}
coords: ['band', 'y', 'x']
dtype:  int16
  [pixel read] 2.049s

shape:      (4, 874, 3519)
size:       12,302,424 pixels (23.5 MB)
throughput: 11 MB/s
range:      [0, 9429]


---
## 4. BBox Crop

관심 영역만 잘라냅니다. Icechunk가 **필요한 byte range만** S3에서 읽습니다.

In [12]:
ds = pq.open_scene(all_scenes[0])
crop_bounds = (128.70, 36.31, 128.74, 36.33)

with Timer("crop + read") as t:
    cropped = pq.crop(ds, crop_bounds)
    crop_data = cropped["data"].values
perf_log["crop"] = t.elapsed

crop_mb = crop_data.nbytes / 1024 / 1024
print(f"원본:  {dict(ds['data'].sizes)}")
print(f"Crop:  {dict(cropped['data'].sizes)}")
print(f"크기:  {crop_mb:.1f} MB")
print(f"시간:  {t.elapsed:.3f}s ({crop_mb/t.elapsed:.0f} MB/s)")

  [crop + read] 0.031s
원본:  {'band': 4, 'y': 874, 'x': 3519}
Crop:  {'band': 4, 'y': 528, 'x': 1282}
크기:  5.2 MB
시간:  0.031s (168 MB/s)


/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(


---
## 5. GeoJSON Polygon Clip

필지 경계(Polygon)로 마스킹합니다. 폴리곤 외부 픽셀은 NaN.

**shapely 2.0 C-level 연산** 사용 (GDAL/rasterio 불필요).

In [22]:
# 농경지 필지 GeoJSON
field_polygon = {
        "type": "MultiPolygon",
        "coordinates": [
          [
            [
              [
                128.75407179142374,
                36.31414901687532
              ],
              [
                128.75528200793664,
                36.313918281300545
              ],
              [
                128.75535312202658,
                36.3137678678695
              ],
              [
                128.7539650752016,
                36.31379571434592
              ],
              [
                128.75407179142374,
                36.31414901687532
              ]
            ]
          ]
        ]
      }

ds = pq.open_scene(all_scenes[0])

# 방법 1: 전체 씬에서 clip
with Timer("clip (전체 씬)") as t:
    clipped = pq.clip(ds, field_polygon)
    clip_data = clipped["data"].values.astype(float)
perf_log["clip"] = t.elapsed

clip_data[clip_data == -999.0] = np.nan
valid_pct = (~np.isnan(clip_data)).sum() / clip_data.size * 100

print(f"Clip 결과: {clip_data.shape}")
print(f"유효 픽셀: {valid_pct:.1f}%  |  NaN: {100-valid_pct:.1f}%")

# 방법 2: crop 먼저 → clip (더 빠름, 불필요한 S3 읽기 절약)
ds = pq.open_scene(all_scenes[0])
bbox = shape(field_polygon).bounds

with Timer("crop → clip") as t:
    cropped = pq.crop(ds, bbox)
    clipped2 = pq.clip(cropped, field_polygon)
    _ = clipped2["data"].values
perf_log["crop_clip"] = t.elapsed

print(f"\ncrop→clip: {t.elapsed:.3f}s (crop 먼저 하면 더 빠름)")

  [clip (전체 씬)] 0.026s
Clip 결과: (4, 10, 44)
유효 픽셀: 64.3%  |  NaN: 35.7%
  [crop → clip] 0.022s

crop→clip: 0.022s (crop 먼저 하면 더 빠름)


/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(


---
## 6. 통계 분석

In [14]:
ds = pq.open_scene(all_scenes[0])

with Timer("statistics") as t:
    for _ in range(100):
        stats = pq.stats(ds)
perf_log["stats"] = t.elapsed / 100

print(f"{'Metric':<10s} {'Value':>10s}")
print(f"{'-'*10} {'-'*10}")
for k, v in stats.items():
    print(f"{k:<10s} {v:>10.2f}")
print(f"\nTime: {perf_log['stats']*1000:.1f}ms")

/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(


  [statistics] 18.359s
Metric          Value
---------- ----------
mean           849.68
std            778.45
min              0.00
max           9429.00
median         638.00
p25            449.00
p75           1087.00

Time: 183.6ms


---
## 7. 필지 시계열 분석

GeoJSON Polygon으로 clip한 영역의 시간별 평균값 변화를 추적합니다.

In [15]:
# 한 줄로 시계열!
with Timer("timeseries") as t:
    ts = pq.timeseries(field_polygon)
perf_log["timeseries"] = t.elapsed

print(f"{'Date':<12s} {'Mean':>8s}")
print(f"{'-'*12} {'-'*8}")
for row in ts:
    print(f"{row['date']:<12s} {row['mean']:>8.1f}")
print(f"\n{len(ts)} scenes | {t.elapsed:.2f}s total | {t.elapsed/len(ts)*1000:.0f}ms/scene")

/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.


  [timeseries] 0.369s
Date             Mean
------------ --------
2025-01-01      720.0
2025-04-06      974.3
2025-05-14     1149.3
2025-07-25     1093.2
2025-07-27        0.0
2025-09-03     1442.9
2025-11-14      803.5

7 scenes | 0.37s total | 53ms/scene


/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(


---
## 8. FeatureCollection (다중 필지 비교)

In [16]:
feature_collection = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"id": "field_A", "name": "논 A"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [128.700, 36.310], [128.710, 36.310],
                    [128.710, 36.315], [128.700, 36.315],
                    [128.700, 36.310],
                ]]
            }
        },
        {
            "type": "Feature",
            "properties": {"id": "field_B", "name": "밭 B"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [128.720, 36.315], [128.730, 36.315],
                    [128.730, 36.322], [128.720, 36.322],
                    [128.720, 36.315],
                ]]
            }
        },
        {
            "type": "Feature",
            "properties": {"id": "field_C", "name": "과수원 C"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [128.735, 36.305], [128.745, 36.305],
                    [128.748, 36.312], [128.738, 36.315],
                    [128.732, 36.310], [128.735, 36.305],
                ]]
            }
        },
    ]
}

# 한 줄로 다중 필지 통계!
with Timer("multi-field stats") as t:
    mf_results = pq.multi_field_stats(feature_collection)
perf_log["multi_field"] = t.elapsed

print(f"{'Name':<10s} {'Pixels':>8s} {'Mean':>8s} {'Std':>8s} {'Min':>8s} {'Max':>8s}")
print(f"{'-'*10} {'-'*8} {'-'*8} {'-'*8} {'-'*8} {'-'*8}")
for r in mf_results:
    print(f"{r['name']:<10s} {r['pixels']:>8,} {r['mean']:>8.1f} {r['std']:>8.1f} {r['min']:>8.1f} {r['max']:>8.1f}")

/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(


  [multi-field stats] 0.081s
Name         Pixels     Mean      Std      Min      Max
---------- -------- -------- -------- -------- --------
논 A         169,488    967.3    729.4      0.0   3502.0
밭 B         237,540   1013.8    782.8    414.0   2928.0
과수원 C       379,412   1086.3    649.3    365.0   3838.0


---
## 9. Polygon Clip → COG Export

클립 결과를 Cloud-Optimized GeoTIFF로 내보냅니다. (Tiled + Overviews + DEFLATE)

In [17]:
# 한 줄로 clip → COG export!
out_path = tempfile.mktemp(suffix="_field_clip.tif")

with Timer("clip → COG export") as t:
    pq.clip_to_cog(all_scenes[0], field_polygon, out_path)
perf_log["export_clip"] = t.elapsed

print(f"출력: {out_path}")
print(f"크기: {os.path.getsize(out_path) / 1024 / 1024:.1f} MB")
print(f"시간: {t.elapsed:.3f}s")

# COG 검증
import rasterio
with rasterio.open(out_path) as src:
    print(f"\n=== COG 검증 ===")
    print(f"Size:        {src.width} x {src.height}")
    print(f"Bands:       {src.count}")
    print(f"CRS:         {src.crs}")
    print(f"Tiled:       {src.profile.get('tiled', False)}")
    print(f"Overviews:   {src.overviews(1)}")
    print(f"Compression: {src.profile.get('compress', 'none')}")
    d = src.read(1)
    valid = d[d != -999.0]
    print(f"Valid:       {len(valid):,} pixels ({len(valid)/d.size*100:.1f}%)")

/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(


  [clip → COG export] 14.673s
출력: /var/folders/gs/cc6p859x305fw3gtsj4_9sm80000gn/T/tmpe01d6zmm_field_clip.tif
크기: 2.4 MB
시간: 14.673s

=== COG 검증 ===
Size:        962 x 449
Bands:       4
CRS:         EPSG:4326
Tiled:       True
Overviews:   [2, 4, 8, 16]
Compression: deflate
Valid:       287,728 pixels (66.6%)


---
## 10. 다중 씬 배치 Export

In [18]:
output_dir = tempfile.mkdtemp(prefix="pq_batch_export_")

with Timer("batch export") as t:
    for s in all_scenes:
        date = s["acquisition_time"][:10]
        out = os.path.join(output_dir, f"{date}_field.tif")
        pq.clip_to_cog(s, field_polygon, out)
perf_log["batch_export"] = t.elapsed

files = sorted(os.listdir(output_dir))
total_mb = sum(os.path.getsize(os.path.join(output_dir, f)) for f in files) / 1024 / 1024

print(f"파일 수: {len(files)}")
print(f"총 크기: {total_mb:.1f} MB")
print(f"시간:    {t.elapsed:.2f}s ({t.elapsed/len(files):.2f}s/scene)")
for f in files:
    sz = os.path.getsize(os.path.join(output_dir, f)) / 1024 / 1024
    print(f"  {f} ({sz:.1f} MB)")

/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.


  [batch export] 3.213s
파일 수: 7
총 크기: 14.6 MB
시간:    3.21s (0.46s/scene)
  2025-01-01_field.tif (2.5 MB)
  2025-04-06_field.tif (2.4 MB)
  2025-05-14_field.tif (2.4 MB)
  2025-07-25_field.tif (2.4 MB)
  2025-07-27_field.tif (0.0 MB)
  2025-09-03_field.tif (2.4 MB)
  2025-11-14_field.tif (2.4 MB)


---
## 11. NDVI 계산 + PNG 렌더링

필지 clip → NDVI 계산 → Colormap 적용 → PNG 이미지 생성. **matplotlib 없이 PIL만으로** 렌더링합니다.

폴리곤 외부는 **투명** (alpha=0) 처리되어 지도 위에 오버레이 가능합니다.

In [23]:
from IPython.display import display, Image as IPImage

# 한 줄로 NDVI PNG!
with Timer("clip → NDVI → PNG") as t:
    png_ndvi = pq.clip_to_png(all_scenes[0], field_polygon, expression="ndvi")
perf_log["ndvi_png"] = t.elapsed

print(f"NDVI PNG: {len(png_ndvi)/1024:.0f} KB | {t.elapsed*1000:.0f}ms")
display(IPImage(data=png_ndvi))

  [clip → NDVI → PNG] 0.049s
NDVI PNG: 1 KB | 49ms


/Users/sonhoyoung/PycharmProjects/pixelquery/.venv/lib/python3.12/site-packages/virtual_tiff/imagecodecs.py:64: UserWarning: Imagecodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  warn(
/Users/sonhoyoung/PycharmProjects/pixelquery/pixelquery/io/s3_client.py:639: RuntimeWarning: invalid value encountered in cast
  indices = (norm * 255).astype(np.uint8)


In [ ]:
# 단일 밴드 렌더링 (viridis colormap)
with Timer("clip → band0 → PNG") as t:
    png_band = pq.clip_to_png(all_scenes[0], field_polygon, expression="band", band=0, colormap="viridis")
perf_log["band_png"] = t.elapsed

print(f"Band0 PNG: {len(png_band)/1024:.0f} KB | {t.elapsed*1000:.0f}ms")
display(IPImage(data=png_band))

In [ ]:
# 3가지 컬러맵 비교
print("=== Colormap 비교 ===\n")

ds = pq.open_scene(all_scenes[0])
bbox = shape(field_polygon).bounds
cropped = pq.crop(ds, bbox)
clipped = pq.clip(cropped, field_polygon)
ndvi = pq.ndvi(clipped)

for cmap in ["rdylgn", "viridis", "inferno"]:
    with Timer(f"  {cmap}") as t:
        png = pq.render_png(ndvi, colormap=cmap, vmin=-0.2, vmax=0.8)
    print(f"  {cmap:8s}: {len(png)/1024:.0f} KB")
    display(IPImage(data=png))

---
## 12. 시계열 NDVI PNG (시간에 따른 식생 변화)

같은 필지의 NDVI를 날짜별로 렌더링하여 식생 변화를 시각적으로 비교합니다.

In [ ]:
# 시계열 NDVI PNG 렌더링
sorted_scenes = sorted(all_scenes, key=lambda s: s["acquisition_time"])

with Timer("timeseries NDVI PNG") as t:
    for s in sorted_scenes:
        date = s["acquisition_time"][:10]
        png = pq.clip_to_png(s, field_polygon, expression="ndvi")
        print(f"  {date}: {len(png)/1024:.0f} KB")
        display(IPImage(data=png))
perf_log["ts_ndvi_png"] = t.elapsed

print(f"\n{len(sorted_scenes)} scenes | {t.elapsed:.2f}s total | {t.elapsed/len(sorted_scenes)*1000:.0f}ms/scene")

---
## 13. Performance Summary

In [ ]:
def fmt(val):
    if val < 0.001:
        return f"{val*1_000_000:.0f}us"
    elif val < 1:
        return f"{val*1000:.1f}ms"
    else:
        return f"{val:.2f}s"

print("=" * 55)
print("  PixelQuery S3 Performance Summary")
print("=" * 55)
print(f"  {'Operation':<35s} {'Time':>10s}")
print(f"  {'-'*35} {'-'*10}")

rows = [
    ("Client init",                    perf_log.get("client_init", 0)),
    ("Ingest (per COG)",               perf_log.get("ingest_per_cog", 0)),
    ("list_scenes (all)",              perf_log.get("list_all", 0)),
    ("list_scenes (time filter)",      perf_log.get("list_time", 0)),
    ("list_scenes (spatial filter)",   perf_log.get("list_spatial", 0)),
    ("list_scenes (time+spatial)",     perf_log.get("list_combined", 0)),
    ("open_scene (lazy)",              perf_log.get("open_lazy", 0)),
    ("Full pixel read (S3)",           perf_log.get("full_read", 0)),
    ("Crop (BBox)",                    perf_log.get("crop", 0)),
    ("Clip (polygon)",                 perf_log.get("clip", 0)),
    ("Crop + Clip",                    perf_log.get("crop_clip", 0)),
    ("Statistics (7 metrics)",         perf_log.get("stats", 0)),
    (f"Timeseries ({len(all_scenes)} scenes)", perf_log.get("timeseries", 0)),
    ("Multi-field (3 polygons)",       perf_log.get("multi_field", 0)),
    ("Clip → COG export",             perf_log.get("export_clip", 0)),
    (f"Batch export ({len(all_scenes)} COGs)", perf_log.get("batch_export", 0)),
    ("--- Rendering ---",             0),
    ("Clip → NDVI → PNG",             perf_log.get("ndvi_png", 0)),
    ("Clip → Band → PNG",             perf_log.get("band_png", 0)),
    (f"TS NDVI PNG ({len(all_scenes)} scenes)", perf_log.get("ts_ndvi_png", 0)),
]

for label, val in rows:
    if label.startswith("---"):
        print(f"  {label}")
    else:
        print(f"  {label:<35s} {fmt(val):>10s}")

print("=" * 55)

---
## API 요약

```python
from pixelquery.io.s3_client import PixelQueryS3

# 클라이언트 생성 (Icechunk + VCC + Registry 자동 설정)
pq = PixelQueryS3(
    bucket="my-bucket",
    endpoint_url="http://localhost:9000",  # MinIO
    access_key_id="...",
    secret_access_key="...",
)

# COG 인제스트
pq.ingest_cogs("arps/", band_names=["blue", "green", "red", "nir"])

# 시공간 검색
scenes = pq.list_scenes(
    time_range=("2025-01-01", "2025-06-01"),
    bounds=(minx, miny, maxx, maxy),
)

# 씬 열기 (lazy)
ds = pq.open_scene(scenes[0])

# BBox crop
cropped = pq.crop(ds, (minx, miny, maxx, maxy))

# Polygon clip (GeoJSON dict)
clipped = pq.clip(ds, geojson_polygon)

# 통계
stats = pq.stats(ds)  # → {mean, std, min, max, median, p25, p75}

# 시계열 (폴리곤 기반)
ts = pq.timeseries(polygon, time_range=("2025-01-01", "2025-12-31"))

# 다중 필지 비교
results = pq.multi_field_stats(feature_collection)

# COG 내보내기 (clip 포함, 한 줄!)
pq.clip_to_cog(scenes[0], polygon, "output.tif")

# --- 렌더링 (matplotlib 불필요, PIL만 사용) ---

# NDVI 계산
ndvi = pq.ndvi(clipped)  # → 2D float32 array

# 2D array → PNG (colormap: rdylgn, viridis, inferno)
png_bytes = pq.render_png(ndvi, colormap="rdylgn", vmin=-0.2, vmax=0.8)

# 한 줄로 clip → NDVI → PNG
png = pq.clip_to_png(scenes[0], polygon, expression="ndvi")

# 단일 밴드 렌더링
png = pq.clip_to_png(scenes[0], polygon, expression="band", band=0, colormap="viridis")

# 픽셀 읽기 (이때 S3 접근)
data = ds["data"].values
```